# Training Scaffold

In [48]:
# Cell 1: imports & basic config
import os
from pathlib import Path
import json
import math
import random
from tqdm.notebook import tqdm
from typing import Tuple, List, Optional, Dict, Any

import numpy as np
import pandas as pd

from PIL import Image, UnidentifiedImageError
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms as T
import torchvision.models as models


# device selection: prefer mps on mac, else cuda, else cpu
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Device:", device)


Device: mps


# 0. Reproducibility & I/O safeguards


In [28]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    

set_seed(42)
cv2.setNumThreads(0)  # avoid OpenCV thread conflicts with PyTorch DataLoader workers

# Dataset (robust, no augmentation; sigma scales with size)

In [29]:
class WheelHeatmapDataset(Dataset):
    def __init__(self, json_path: str, img_dir: str, img_size: int = 128, sigma_ratio: float = 0.02, max_skip: int = 10):
        with open(json_path, "r") as f:
            self.labels = json.load(f)
        self.img_dir = Path(img_dir)
        self.img_size = int(img_size)
        self.sigma = max(1.0, sigma_ratio * self.img_size)  # scale with size
        self.max_skip = max_skip

        self.transform = T.Compose([
            T.Resize((self.img_size, self.img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int):
        tries = 0
        n = len(self.labels)
        while tries < self.max_skip:
            entry = self.labels[idx]
            img_path = self.img_dir / entry["filename"]
            try:
                img = Image.open(img_path).convert("RGB")
            except (FileNotFoundError, UnidentifiedImageError, OSError) as e:
                print(f"[WARN] Skipping unreadable image: {img_path} ({e})")
                idx = (idx + 1) % n
                tries += 1
                continue

            img_t = self.transform(img)

            # Build heatmap target and presence flag
            heatmap = np.zeros((self.img_size, self.img_size), dtype=np.float32)
            presence = 1.0 if float(entry.get("is_present", 1.0)) == 1.0 else 0.0
            if presence == 1.0:
                cx = int(entry["center"][0] * self.img_size)
                cy = int(entry["center"][1] * self.img_size)
                heatmap = self._draw_gaussian(heatmap, (cx, cy), self.sigma)

            heat_t = torch.tensor(heatmap, dtype=torch.float32).unsqueeze(0)
            presence_t = torch.tensor(presence, dtype=torch.float32)

            return {
                "image": img_t,
                "heatmap": heat_t,
                "presence": presence_t,
                "filename": entry["filename"],
            }

        raise RuntimeError(f"Too many consecutive bad samples encountered starting at index {idx}.")

    @staticmethod
    def _draw_gaussian(heatmap: np.ndarray, center: Tuple[int, int], sigma: float) -> np.ndarray:
        x0, y0 = center
        h, w = heatmap.shape
        xs = np.arange(w, dtype=np.float32)
        ys = np.arange(h, dtype=np.float32)[:, None]
        g = np.exp(-((xs - x0) ** 2 + (ys - y0) ** 2) / (2 * sigma ** 2))
        return np.maximum(heatmap, g.astype(np.float32))


# 2. Model: ResNet18 encoder + U-Net skips + bilinear upsampling
#     Multi-task heads: heatmap + presence

In [34]:
class UNetResNet18Multi(nn.Module):
    def __init__(self, pretrained: bool = True, dropout_p: float = 0.1):
        super().__init__()
        backbone = models.resnet18(pretrained=pretrained)

        # Encoder (keep spatial tensors for skips)
        self.enc1 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # /2
        self.enc2 = nn.Sequential(backbone.maxpool, backbone.layer1)            # /4
        self.enc3 = backbone.layer2                                             # /8
        self.enc4 = backbone.layer3                                             # /16
        self.enc5 = backbone.layer4                                             # /32

        # Decoder blocks: Upsample (bilinear) + Conv-BN-ReLU-Dropout
        def up_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout_p),
            )

        # Channels from ResNet18 stages: e1=64, e2=64, e3=128, e4=256, e5=512
        self.up5 = up_block(512, 256)                  # -> /16
        self.up4 = up_block(256 + 256, 128)            # concat e4
        self.up3 = up_block(128 + 128, 64)             # concat e3
        self.up2 = up_block(64 + 64, 64)               # concat e2
        self.up1 = up_block(64 + 64, 32)               # concat e1

        # Heatmap head
        self.head_heatmap = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=1),
        )

        # Presence head (global pooling on deepest features)
        self.presence_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.presence_fc = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_p),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        e1 = self.enc1(x)     # C64, /2
        e2 = self.enc2(e1)    # C64, /4
        e3 = self.enc3(e2)    # C128, /8
        e4 = self.enc4(e3)    # C256, /16
        e5 = self.enc5(e4)    # C512, /32

        # presence
        pooled = self.presence_pool(e5).flatten(1)
        presence_logit = self.presence_fc(pooled).squeeze(1)

        # decoder with skips
        d5 = self.up5(e5)             # /16
        d4 = self.up4(torch.cat([d5, e4], dim=1))   # /8
        d3 = self.up3(torch.cat([d4, e3], dim=1))   # /4
        d2 = self.up2(torch.cat([d3, e2], dim=1))   # /2
        d1 = self.up1(torch.cat([d2, e1], dim=1))   # /1

        heatmap_logits = self.head_heatmap(d1)      # (B,1,H,W) same as input

        # Safety: ensure same HxW as input
        if heatmap_logits.shape[2:] != x.shape[2:]:
            heatmap_logits = F.interpolate(heatmap_logits, size=x.shape[2:], mode="bilinear", align_corners=False)

        return {"heatmap_logits": heatmap_logits, "presence_logit": presence_logit}


# 3. Losses, metrics, utilities

In [35]:
def multi_task_loss(preds: Dict[str, torch.Tensor],
                    targets: Dict[str, torch.Tensor],
                    heatmap_weight: float = 1.0,
                    presence_weight: float = 1.0) -> Tuple[torch.Tensor, Dict[str, float]]:
    heatmap_logits = preds["heatmap_logits"]
    presence_logit = preds["presence_logit"]
    target_heat = targets["heatmap"].to(heatmap_logits.device)
    target_presence = targets["presence"].to(presence_logit.device)

    # presence: BCE with logits
    presence_loss = F.binary_cross_entropy_with_logits(presence_logit, target_presence)

    # heatmap: sigmoid + MSE; mask by presence
    pred_heat = torch.sigmoid(heatmap_logits)
    mask = target_presence.view(-1, 1, 1, 1)  # broadcast
    if mask.sum() > 0:
        mse = (pred_heat - target_heat).pow(2)
        heatmap_loss = (mse * mask).sum() / (mask.sum() * target_heat[0].numel() + 1e-12)
    else:
        heatmap_loss = torch.tensor(0.0, device=heatmap_logits.device)

    total = presence_weight * presence_loss + heatmap_weight * heatmap_loss
    return total, {"presence_loss": float(presence_loss.item()),
                   "heatmap_loss": float(heatmap_loss.item())}

def soft_argmax_2d_numpy(heatmap: np.ndarray) -> Tuple[float, float]:
    h, w = heatmap.shape
    flat = heatmap.reshape(-1)
    flat -= flat.max()
    prob = np.exp(flat)
    prob /= (prob.sum() + 1e-12)
    xs = np.arange(w, dtype=np.float32)
    ys = np.arange(h, dtype=np.float32)
    xv, yv = np.meshgrid(xs, ys)
    x = float((prob * xv.reshape(-1)).sum())
    y = float((prob * yv.reshape(-1)).sum())
    return x, y

def compute_metrics(preds: Dict[str, torch.Tensor],
                    targets: Dict[str, torch.Tensor],
                    px_thresholds: List[int] = [2, 5, 10]) -> Dict[str, Any]:
    pred_presence_prob = torch.sigmoid(preds["presence_logit"]).detach().cpu().numpy()
    pred_presence = (pred_presence_prob >= 0.5).astype(np.uint8)
    target_presence = targets["presence"].detach().cpu().numpy().astype(np.uint8)

    TP = int(((pred_presence == 1) & (target_presence == 1)).sum())
    FP = int(((pred_presence == 1) & (target_presence == 0)).sum())
    FN = int(((pred_presence == 0) & (target_presence == 1)).sum())
    precision = TP / (TP + FP + 1e-12)
    recall = TP / (TP + FN + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    # localization metrics on samples where both GT and pred say "present"
    pred_hm = torch.sigmoid(preds["heatmap_logits"]).detach().cpu().numpy()
    target_hm = targets["heatmap"].detach().cpu().numpy()
    errors = []
    for i in range(len(pred_hm)):
        if target_presence[i] == 1 and pred_presence[i] == 1:
            px, py = soft_argmax_2d_numpy(pred_hm[i, 0])
            gx, gy = soft_argmax_2d_numpy(target_hm[i, 0])
            errors.append(math.hypot(px - gx, py - gy))

    out = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_px_err": float(np.mean(errors)) if errors else float("nan"),
        "median_px_err": float(np.median(errors)) if errors else float("nan"),
        "n_loc_samples": int(len(errors)),
    }
    for thr in px_thresholds:
        out[f"pct_within_{thr}px"] = float(np.mean(np.array(errors) <= thr) * 100) if errors else float("nan")
    return out

def colorize_heatmap(hm: np.ndarray) -> np.ndarray:
    hm = hm.copy()
    mmin, mmax = hm.min(), hm.max()
    if mmax > mmin:
        hm = (hm - mmin) / (mmax - mmin)
    else:
        hm = np.zeros_like(hm)
    return cv2.applyColorMap((hm * 255).astype(np.uint8), cv2.COLORMAP_JET)

def draw_overlay(img_bgr: np.ndarray, gt_xy: Tuple[int, int], pred_xy: Tuple[int, int], err: float) -> np.ndarray:
    vis = img_bgr.copy()
    if gt_xy is not None:
        cv2.circle(vis, gt_xy, 4, (0, 0, 255), -1)  # GT red
    if pred_xy is not None:
        cv2.circle(vis, pred_xy, 4, (0, 255, 0), 2)  # Pred green
    cv2.putText(vis, f"err={err:.2f}px", (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
    return vis


# 4. Training loop with AMP, LR scheduler, Early stopping

In [36]:
def train_model(model: nn.Module,
                train_loader: DataLoader,
                val_loader: DataLoader,
                epochs: int = 30,
                lr: float = 1e-3,
                device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
                patience: int = 6,
                heatmap_weight: float = 1.0,
                presence_weight: float = 1.0,
                out_dir: Path = Path("out")) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    scaler = GradScaler(enabled=device.type == "cuda")

    best_val = float("inf")
    bad_epochs = 0
    best_path = out_dir / "best_model.pth"

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for batch in train_loader:
            imgs = batch["image"].to(device, non_blocking=True)
            heat = batch["heatmap"].to(device, non_blocking=True)
            pres = batch["presence"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=device.type == "cuda"):
                preds = model(imgs)
                loss, _ = multi_task_loss(preds, {"heatmap": heat, "presence": pres},
                                          heatmap_weight=heatmap_weight, presence_weight=presence_weight)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_losses.append(float(loss.item()))

        # Validation
        model.eval()
        val_losses = []
        val_metrics_accum = []
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch["image"].to(device, non_blocking=True)
                heat = batch["heatmap"].to(device, non_blocking=True)
                pres = batch["presence"].to(device, non_blocking=True)
                with autocast(enabled=device.type == "cuda"):
                    preds = model(imgs)
                    loss, _ = multi_task_loss(preds, {"heatmap": heat, "presence": pres},
                                              heatmap_weight=heatmap_weight, presence_weight=presence_weight)
                val_losses.append(float(loss.item()))
                val_metrics_accum.append(compute_metrics(preds, {"heatmap": heat, "presence": pres}))

        # Aggregate metrics
        train_loss = float(np.mean(train_losses)) if train_losses else float("inf")
        val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
        scheduler.step(val_loss)

        # Average val metrics
        if val_metrics_accum:
            keys = val_metrics_accum[0].keys()
            val_metrics = {k: float(np.nanmean([m[k] for m in val_metrics_accum])) for k in keys}
        else:
            val_metrics = {}

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"Train {train_loss:.4f} | Val {val_loss:.4f} | "
              f"F1 {val_metrics.get('f1', float('nan')):.3f} | "
              f"MeanErr {val_metrics.get('mean_px_err', float('nan')):.2f}px | "
              f"<=5px {val_metrics.get('pct_within_5px', float('nan')):.1f}%")

        # Early stopping on val_loss
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            bad_epochs = 0
            torch.save({"model_state": model.state_dict(),
                        "epoch": epoch}, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print("Early stopping triggered.")
                break

    print(f"Best model saved to: {best_path}")
    return best_path


# 5. Evaluation + visualization

In [37]:
def evaluate_and_visualize(model: nn.Module,
                           loader: DataLoader,
                           img_dir: Path,
                           device: torch.device,
                           out_dir: Path = Path("out/eval"),
                           worst_n: int = 30) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir = out_dir / "overlays"; overlay_dir.mkdir(exist_ok=True)
    heatmap_dir = out_dir / "heatmaps"; heatmap_dir.mkdir(exist_ok=True)

    model.eval()
    all_rows = []
    errors = []

    with torch.no_grad():
        for batch in loader:
            imgs = batch["image"].to(device)
            heat = batch["heatmap"].to(device)
            pres = batch["presence"].to(device)
            fnames = batch["filename"]

            preds = model(imgs)
            metrics = compute_metrics(preds, {"heatmap": heat, "presence": pres})

            pred_presence = (torch.sigmoid(preds["presence_logit"]) >= 0.5).cpu().numpy().astype(np.uint8)
            pred_hm = torch.sigmoid(preds["heatmap_logits"]).cpu().numpy()
            gt_hm = heat.cpu().numpy()

            for i, fname in enumerate(fnames):
                # Load the original image and resize to model input size for alignment
                img_bgr = cv2.imread(str(img_dir / fname))
                if img_bgr is None:
                    continue
                H, W = gt_hm[i, 0].shape
                img_bgr = cv2.resize(img_bgr, (W, H), interpolation=cv2.INTER_LINEAR)

                # centers
                gx, gy = soft_argmax_2d_numpy(gt_hm[i, 0])
                px, py = soft_argmax_2d_numpy(pred_hm[i, 0])

                err = float(math.hypot(px - gx, py - gy))
                errors.append(err)

                vis = draw_overlay(img_bgr, (int(gx), int(gy)) if pres[i].item() == 1.0 else None,
                                   (int(px), int(py)) if pred_presence[i] == 1 else None, err)
                cv2.imwrite(str(overlay_dir / f"overlay_{Path(fname).name}"), vis)

                # heatmap blend
                hm_color = colorize_heatmap(pred_hm[i, 0])
                hm_overlay = cv2.addWeighted(img_bgr, 1.0, hm_color, 0.45, 0)
                cv2.circle(hm_overlay, (int(gx), int(gy)), 3, (0, 0, 255), -1)
                cv2.circle(hm_overlay, (int(px), int(py)), 3, (0, 255, 0), 2)
                cv2.imwrite(str(heatmap_dir / f"hm_{Path(fname).name}"), hm_overlay)

                all_rows.append({
                    "filename": fname,
                    "gt_x": gx, "gt_y": gy,
                    "pred_x": px, "pred_y": py,
                    "err_px": err,
                    "pred_presence": int(pred_presence[i]),
                    "gt_presence": int(pres[i].item()),
                })

    # Sort worst cases
    if errors:
        idx_sorted = np.argsort(errors)[::-1][:min(worst_n, len(errors))]
        worst_dir = out_dir / "worst_overlays"; worst_dir.mkdir(exist_ok=True)
        for idx in idx_sorted:
            fname = all_rows[idx]["filename"]
            src = overlay_dir / f"overlay_{Path(fname).name}"
            if src.exists():
                dst = worst_dir / f"{errors[idx]:.2f}px__{Path(fname).name}"
                img = cv2.imread(str(src))
                cv2.imwrite(str(dst), img)

    # Aggregate metrics over entire loader
    # Reuse compute_metrics on the fly: simple aggregate already printed during training
    mean_err = float(np.mean(errors)) if errors else float("nan")
    median_err = float(np.median(errors)) if errors else float("nan")
    pct2 = float(np.mean(np.array(errors) <= 2) * 100) if errors else float("nan")
    pct5 = float(np.mean(np.array(errors) <= 5) * 100) if errors else float("nan")
    pct10 = float(np.mean(np.array(errors) <= 10) * 100) if errors else float("nan")

    print(f"Eval overlays: {overlay_dir}")
    print(f"Heatmap overlays: {heatmap_dir}")
    print(f"Mean err: {mean_err:.2f}px | Median: {median_err:.2f}px | <=2px {pct2:.1f}% | <=5px {pct5:.1f}% | <=10px {pct10:.1f}%")
    return {"mean_err": mean_err, "median_err": median_err, "pct2": pct2, "pct5": pct5, "pct10": pct10}


# 6. Usage

In [47]:
if __name__ == "__main__":
    base_dir = Path.cwd()
    img_dir = base_dir / "labeled_example_data"
    labels_path = img_dir / "labels.json"

    dataset = WheelHeatmapDataset(labels_path, img_dir, img_size=128, sigma_ratio=0.02, max_skip=10)
    n = len(dataset)
    train_size = int(0.8 * n)
    val_size = n - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size],
                                                     generator=torch.Generator().manual_seed(42))

    # DataLoaders with multiprocessing safeguards

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=0)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNetResNet18Multi(pretrained=True)

    best_ckpt = train_model(model, train_loader, val_loader,
                            epochs=40, lr=1e-3, device=device,
                            patience=6, heatmap_weight=1.0, presence_weight=1.0,
                            out_dir=Path("out"))

    # Load best and run evaluation + visualization on validation set
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)

    print("Running evaluation + visualization on validation split...")
    evaluate_and_visualize(model, val_loader, img_dir=img_dir, device=device, out_dir=Path("out/eval"), worst_n=30)

/var/folders/8m/p7s3fq9j1y55c3pc57sg7z2r0000gn/T/ipykernel_58112/3533082571.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=device.type == "cuda")
/var/folders/8m/p7s3fq9j1y55c3pc57sg7z2r0000gn/T/ipykernel_58112/3533082571.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=device.type == "cuda"):
/var/folders/8m/p7s3fq9j1y55c3pc57sg7z2r0000gn/T/ipykernel_58112/3533082571.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=device.type == "cuda"):


Epoch 01/40 | Train 0.0415 | Val 0.0014 | F1 1.000 | MeanErr 0.08px | <=5px 100.0%
Epoch 02/40 | Train 0.0013 | Val 0.0012 | F1 1.000 | MeanErr 0.07px | <=5px 100.0%
Epoch 03/40 | Train 0.0011 | Val 0.0010 | F1 1.000 | MeanErr 0.08px | <=5px 100.0%
Epoch 04/40 | Train 0.0009 | Val 0.0008 | F1 1.000 | MeanErr 0.06px | <=5px 100.0%
Epoch 05/40 | Train 0.0008 | Val 0.0007 | F1 1.000 | MeanErr 0.05px | <=5px 100.0%
Epoch 06/40 | Train 0.0007 | Val 0.0006 | F1 1.000 | MeanErr 0.06px | <=5px 100.0%
Epoch 07/40 | Train 0.0006 | Val 0.0005 | F1 1.000 | MeanErr 0.05px | <=5px 100.0%
Epoch 08/40 | Train 0.0005 | Val 0.0005 | F1 1.000 | MeanErr 0.04px | <=5px 100.0%
Epoch 09/40 | Train 0.0005 | Val 0.0005 | F1 1.000 | MeanErr 0.05px | <=5px 100.0%
Epoch 10/40 | Train 0.0004 | Val 0.0005 | F1 1.000 | MeanErr 0.05px | <=5px 100.0%
Epoch 11/40 | Train 0.0004 | Val 0.0005 | F1 1.000 | MeanErr 0.05px | <=5px 100.0%
Epoch 12/40 | Train 0.0004 | Val 0.0004 | F1 1.000 | MeanErr 0.04px | <=5px 100.0%
Epoc

In [52]:
import os
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# ===== CONFIG =====
IMG_DIR = "labeled_example_data"  # folder with images
CSV_FILE = "out/heatmap_results.csv"  # or regression_results.csv
OUTPUT_DIR = "out/pred_vs_gt"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== LOAD RESULTS =====
df = pd.read_csv(CSV_FILE)

for _, row in df.iterrows():
    img_path = os.path.join(IMG_DIR, row["filename"])
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: could not read {img_path}")
        continue

    # Draw GT in green
    if not pd.isna(row["gt_x"]) and not pd.isna(row["gt_y"]):
        cv2.circle(img, (int(row["gt_x"]), int(row["gt_y"])), 5, (0, 255, 0), -1)

    # Draw prediction in red
    if not pd.isna(row["pred_x"]) and not pd.isna(row["pred_y"]):
        cv2.circle(img, (int(row["pred_x"]), int(row["pred_y"])), 5, (0, 0, 255), -1)

    # Save overlay
    out_path = os.path.join(OUTPUT_DIR, row["filename"])
    cv2.imwrite(out_path, img)

    # Optional: also show inline
    # plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    # plt.title(row["filename"])
    # plt.show()

print(f"Saved overlays to: {OUTPUT_DIR}")

FileNotFoundError: [Errno 2] No such file or directory: 'out/heatmap_results.csv'